# Scenario

TheLook, a hypothetical eCommerce clothing retailer, stores data on customers, products, orders, logistics, web events, and digital marketing campaigns in BigQuery. The company wants to migrate some of this data to Apache Iceberg and then leverage the team's existing SQL and PySpark expertise to analyze this data using Apache Spark.

To avoid manual infrastructure provisioning or tuning for Spark, TheLook seeks an auto-scaling solution that allows them to focus on workloads rather than cluster management. Additionally, they want to minimize the effort required to integrate Spark and BigQuery while staying within the BigQuery Studio environment, possibly using BigQuery notebooks.

As an example, let's understand how they can perform following analysis using Apache Spark:

# Goal

Predict whether a user will make a purchase by building a Logistic Regression Classifier using PySpark and leverage BQ Studio's native Integration and AI-features for exploring the data


# **Step 0: Setup**

Enable the necessary APIs.

In [208]:
!gcloud services enable dataproc.googleapis.com

In [192]:
!sudo apt-get install python3.11

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.11 is already the newest version (3.11.13-1+jammy1).
The following packages were automatically installed and are no longer required:
  r-cran-colorspace r-cran-munsell
Use 'sudo apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 48 not upgraded.


Configure a project id and location.

In [194]:
!python --version

Python 3.10.12


In [177]:
PROJECT_ID = "iceberg-demo-2025-2" # @param {type:"string"}

LOCATION = "US" # @param {type:"string"}

Create a [Cloud Storage bucket](https://cloud.google.com/storage/docs/creating-buckets?utm_campaign=CDR_0x225cfd13_default_b407565440&utm_source=external&utm_medium=web).


In [209]:
from google.cloud import storage

BUCKET_NAME = "iceberg-demo-2025-2" # @param {type:"string"}

storage_client = storage.Client(project=PROJECT_ID)
bucket_obj = storage_client.create_bucket(BUCKET_NAME, location=LOCATION)

Conflict: 409 POST https://storage.googleapis.com/storage/v1/b?project=bmiro-external&prettyPrint=false: Your previous request to create the named bucket succeeded and you already own it.

In [49]:
?utm_campaign=CDR_0x225cfd13_default_b407565440&utm_source=external&utm_medium=web

Object `utm_campaign=CDR_0x225cfd13_default_b407565440&utm_source=external&utm_medium=web` not found.


Create a [BigQuery connection](https://cloud.google.com/bigquery/docs/create-cloud-resource-connection?utm_campaign=CDR_0x225cfd13_default_b407565440&utm_source=external&utm_medium=web) to Cloud resources.

In [211]:
from google.cloud import bigquery_connection_v1 as bq_connection
from google.cloud.bigquery_connection_v1 import types

bq_conn_client = bq_connection.ConnectionServiceClient()
parent_path = bq_conn_client.common_location_path(PROJECT_ID, LOCATION)

cloud_resource_properties = types.CloudResourceProperties()
connection = types.Connection(cloud_resource=cloud_resource_properties)

request = types.CreateConnectionRequest(
    parent=parent_path,
    connection=connection,
    connection_id="my_connection5",
)

conn_response = bq_conn_client.create_connection(request=request)

AlreadyExists: 409 Already Exists: Connection projects/176581813365/locations/us/connections/my_connection5

Give the service account attached to your connection the necessary permissions.

In [212]:
policy = bucket_obj.get_iam_policy(requested_policy_version=3)

member = f"serviceAccount:{conn_response.cloud_resource.service_account_id}"
role_to_add = "roles/storage.admin"

policy.bindings.append(
    {
        "role": role_to_add,
        "members": [member],
    }
)

bucket_obj.set_iam_policy(policy)

Create a BigQuery dataset.

In [213]:
from google.cloud import bigquery

client = bigquery.Client()

dataset = bigquery.Dataset(f"{PROJECT_ID}.my_dataset2")

dataset.location = LOCATION

dataset = client.create_dataset(dataset)

Conflict: 409 POST https://bigquery.googleapis.com/bigquery/v2/projects/bmiro-external/datasets?prettyPrint=false: Already Exists: Dataset bmiro-external:my_dataset2

In [214]:
# Create Iceberg tables
%%bigquery
CREATE TABLE my_dataset2.users2 (
  id INTEGER,
  first_name STRING,
  last_name STRING,
  email STRING,
  age INTEGER,
  gender STRING,
  state STRING,
  street_address STRING,
  postal_code STRING,
  city STRING,
  country STRING,
  latitude FLOAT64,
  longitude FLOAT64,
  traffic_source STRING,
  created_at TIMESTAMP
)
WITH CONNECTION `us.my_connection5`
OPTIONS (
  file_format = "PARQUET",
  table_format = "ICEBERG",
  storage_uri = "gs://iceberg-demo-2025-2/iceberg-demo/users2"
)


Executing query with job ID: 4f4364b6-143d-403c-9f23-978000d857fc
Query executing: 0.33s


ERROR:
 409 GET https://bigquery.googleapis.com/bigquery/v2/projects/bmiro-external/queries/4f4364b6-143d-403c-9f23-978000d857fc?maxResults=0&location=US&prettyPrint=false: Already Exists: Table bmiro-external:my_dataset2.users2

Location: US
Job ID: 4f4364b6-143d-403c-9f23-978000d857fc



In [215]:
# Create Iceberg tables
%%bigquery
CREATE TABLE my_dataset2.order_items (
  id INTEGER,
  order_id INTEGER,
  user_id INTEGER,
  product_id INTEGER,
  inventory_item_id INTEGER,
  status STRING,
  created_at TIMESTAMP,
  shipped_at TIMESTAMP,
  delivered_at TIMESTAMP,
  returned_at TIMESTAMP,
  sale_price FLOAT64
)
WITH CONNECTION `us.my_connection5`
OPTIONS (
  file_format = "PARQUET",
  table_format = "ICEBERG",
  storage_uri = "gs://iceberg-demo-2025-2/iceberg-demo/order_items"
)


Executing query with job ID: 19d1a5d7-4760-4f09-b666-6d723a21f010
Query executing: 0.35s


ERROR:
 409 GET https://bigquery.googleapis.com/bigquery/v2/projects/bmiro-external/queries/19d1a5d7-4760-4f09-b666-6d723a21f010?maxResults=0&location=US&prettyPrint=false: Already Exists: Table bmiro-external:my_dataset2.order_items

Location: US
Job ID: 19d1a5d7-4760-4f09-b666-6d723a21f010



In [216]:
%%bigquery
INSERT INTO my_dataset2.users2
SELECT
  id,
  first_name,
  last_name,
  email,
  age,
  gender,
  state,
  street_address,
  postal_code,
  city,
  country,
  latitude,
  longitude,
  traffic_source,
  created_at,
FROM bigquery-public-data.thelook_ecommerce.users

Query is running:   0%|          |

""


In [217]:
%%bigquery
INSERT INTO my_dataset2.order_items
SELECT * FROM `bigquery-public-data.thelook_ecommerce.order_items`

Query is running:   0%|          |

""


In [222]:
!gcloud storage ls gs://iceberg-demo-2025-2/iceberg-demo/users2/data

gs://iceberg-demo-2025-2/iceberg-demo/users2/data/25bc9bd7-3632-48d7-9e85-a5f3fbd9cb64-0b9bf6772efcb901-f-00000-of-00001.parquet
gs://iceberg-demo-2025-2/iceberg-demo/users2/data/7cc32cd4-516e-461d-b8de-c4fff7825bd2-3c5e723304d146e4-f-00000-of-00001.parquet


In [ ]:
%%bigquery
select * from my_dataset2.users limit 10

# **Step 1: Setup Spark**

*   Set up the Spark environment: It imports necessary
libraries for connecting to Dataproc and using PySpark MLlib Connect.
*   Configure the Dataproc session: It creates and configures a Spark Session with the necessary parameters, providing the spark object for subsequent Spark operations.

This step can also be accomplished in a single line of code below, but we need  customization for our Spark ML code.



```
spark = DataprocSparkSession.builder.getOrCreate()
```





In [207]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

from pyspark.ml.connect.classification import LogisticRegression, LogisticRegressionModel
from pyspark.ml.connect.evaluation import BinaryClassificationEvaluator
from pyspark.ml.connect.feature import StandardScaler
from pyspark.ml.connect.pipeline import Pipeline
import torch
import pickle

spark.stop()
session = Session()
session.runtime_config.properties = {
  'spark.dynamicAllocation.enabled': 'false'
}
session.runtime_config.version = "2.2"
spark = (
    DataprocSparkSession.builder
      .appName("CustomSparkSession")
      .dataprocSessionConfig(session)
      .getOrCreate()
)

Creating Dataproc Session: https://console.cloud.google.com/dataproc/interactive/us-central1/sc-20250703-004336-amx3ha?project=bmiro-external


██████████████████████████████████████████████▉                                 



Dataproc Session was successfully created


/usr/local/lib/python3.10/dist-packages/pyspark/sql/connect/session.py:185: UserWarning: [CANNOT_MODIFY_CONFIG] Cannot modify the value of the Spark config: "spark.dynamicAllocation.enabled".
See also 'https://spark.apache.org/docs/latest/sql-migration-guide.html#ddl-statements'.
  warnings.warn(str(e))


In [186]:
!python --version

Python 3.10.12


In [187]:
spark.sql("Select * from my_dataset2.users2");

# **Step 2: Load Data**

Load the users and order_items table from the publicly available dataset for The Look (https://www.kaggle.com/datasets/mustafakeser4/looker-ecommerce-bigquery-dataset))

This is done simply by loading directly into a Spark Dataframe.

In [146]:
spark.sql("SHOW DATABASES").show()

+--------------+
|     namespace|
+--------------+
|bmiro_external|
|  cymbal_shops|
|    my_dataset|
|   my_dataset2|
|     mydataset|
|   predictions|
|sample_dataset|
|       thelook|
+--------------+



In [145]:
spark.sql("SHOW TABLES in my_dataset2").show()

+--------------------+-----------+-----------+
|           namespace|  tableName|isTemporary|
+--------------------+-----------+-----------+
|`bmiro-external`....|order_items|      false|
|`bmiro-external`....|      users|      false|
|`bmiro-external`....|     users2|      false|
+--------------------+-----------+-----------+



In [188]:
users = spark.sql("select * from my_dataset2.users2")
order_items = spark.sql("select * from my_dataset2.order_items")

In [133]:
users.take(5)

[Row(id=78551, first_name='Shawna', last_name='Hawkins', email='shawnahawkins@example.net', age=28, gender='F', state='Aichi', street_address='204 Alvarez Meadow Apt. 367', postal_code='455-0028', city='Nagoya city', country='Japan', latitude=35.06073258, longitude=136.8742865, traffic_source='Search', created_at=datetime.datetime(2023, 5, 14, 18, 27)),
 Row(id=94799, first_name='Michael', last_name='Logan', email='michaellogan@example.org', age=22, gender='M', state='Alabama', street_address='521 Jennifer Dale Suite 684', postal_code='36609', city='Mobile', country='United States', latitude=30.66140191, longitude=-88.16291857, traffic_source='Email', created_at=datetime.datetime(2024, 12, 28, 15, 13)),
 Row(id=9485, first_name='Christopher', last_name='King', email='christopherking@example.org', age=46, gender='M', state='Alabama', street_address='1154 Gonzalez Pike Suite 185', postal_code='35601', city='Decatur', country='United States', latitude=34.61337915, longitude=-87.00970929, 

# **Step 3: Data Exploration**

Bigquery Studio can leverage Gemini for advanced code completion capabilities ([https://cloud.google.com/bigquery/docs/write-sql-gemini#generate_python_code])(https://) which can use Natual Language to perform exploratory analysis using SQL and even generate PySpark Code for Feature Engineering.

In this example we wan to see the distribution of countries in the user table

Prompt 1: Explore the users table and show the first 10 rows

Prompt 2: Explore the order_items table and show the first 10 rows

Prompt 3: Generate PySpark code to show the top 5 most frequent countries in the users table. Display the country and the number of users from each country. Spark Session is already active and the table is loaded in a dataframe

Prompt 4: Generate PySpark code to find the average sale price of items in the order_items table.Spark Session is already active and the table is loaded in a dataframe

In [149]:
# prompt: Explore the users table and show the first 10 rows using Spark

# NOTE: Pyspark code generation is currently in PREVIEW.
users.show(10)

+-----+-----------+---------+--------------------+---+------+-------+--------------------+-----------+---------------+-------------+------------+------------+--------------+-------------------+
|   id| first_name|last_name|               email|age|gender|  state|      street_address|postal_code|           city|      country|    latitude|   longitude|traffic_source|         created_at|
+-----+-----------+---------+--------------------+---+------+-------+--------------------+-----------+---------------+-------------+------------+------------+--------------+-------------------+
|78551|     Shawna|  Hawkins|shawnahawkins@exa...| 28|     F|  Aichi|204 Alvarez Meado...|   455-0028|    Nagoya city|        Japan| 35.06073258| 136.8742865|        Search|2023-05-14 18:27:00|
|94799|    Michael|    Logan|michaellogan@exam...| 22|     M|Alabama|521 Jennifer Dale...|      36609|         Mobile|United States| 30.66140191|-88.16291857|         Email|2024-12-28 15:13:00|
| 9485|Christopher|     King|c

In [ ]:
# prompt: Explore the order_items table and show the first 10 rows

order_items.show(10)

In [ ]:
spark.version

In [195]:
# prompt: Generate PySpark code to show the top 5 most frequent countries in the users table. Display the country and the number of users from each country. Spark Session is already active and the table is loaded in a dataframe

# NOTE: Pyspark code generation is currently in PREVIEW.
from pyspark.sql.functions import col, count

users.groupBy("country").agg(count("*").alias("user_count")).orderBy(col("user_count").desc()).limit(5).show()

RuntimeError: SparkContext or SparkSession should be created first.

In [196]:
# prompt: Generate PySpark code to show the top 5 most frequent countries in the users table. Display the country and the number of users from each country. Spark Session is already active and the table is loaded in a dataframe

# NOTE: Pyspark code generation is currently in PREVIEW.
from pyspark.sql.functions import col, count, desc

users.groupBy("country") \
    .agg(count("*").alias("user_count")) \
    .orderBy(desc("user_count")) \
    .limit(5) \
    .select("country", "user_count") \
    .show()

RuntimeError: SparkContext or SparkSession should be created first.

In [197]:
# prompt: Generate PySpark code to show the top 5 most frequent countries in the users table. Display the country and the number of users from each country. Spark Session is already active and the table is loaded in a dataframe

from pyspark.sql.functions import col, count, desc

users.groupBy("country") \
    .agg(count("*").alias("user_count")) \
    .orderBy(desc("user_count")) \
    .limit(5) \
    .select("country", "user_count") \
    .show()

RuntimeError: SparkContext or SparkSession should be created first.

In [ ]:
# prompt: Generate PySpark code to show the top 5 most frequent countries in the users table. Display the country and the number of users from each country. Spark Session is already active and the table is loaded in a dataframe

# Assuming 'users' DataFrame is already loaded and SparkSession is active.

users.groupBy("country").count().orderBy("count", ascending=False).limit(5).show()

In [ ]:
# prompt: Generate PySpark code to find the average sale price of items in the order_items table.Spark Session is already active and the table is loaded in a dataframe

# Assuming 'order_items' DataFrame is already loaded and SparkSession is active.

order_items.agg({"sale_price": "avg"}).show()

#**Step 4: Feature Engineering**

In this step, we derive two key columns from the input data:

**Creation of features column**:
It combines user attributes (age, hashed categorical features) into a numerical array, preparing them for a machine learning model.

**Generation of label column:**
It creates a binary target variable indicating whether a user has made a purchase or not, derived from order information.

In [189]:
# Load BigQuery dataset with feature engineering in SQL
dataset = spark.sql("""
SELECT
  ARRAY(
        CAST(u.age AS DOUBLE),
        CAST(hash(u.country) AS BIGINT) * 1.0,
        CAST(hash(u.gender) AS BIGINT) * 1.0,
        CAST(hash(u.traffic_source) AS BIGINT) * 1.0
    ) AS features,
    CASE WHEN COALESCE(SUM(oi.sale_price), 0) > 0 THEN 1 ELSE 0 END AS label
FROM my_dataset2.users2 AS u
LEFT JOIN my_dataset2.order_items AS oi
ON u.id = oi.user_id
GROUP BY u.id, u.age, u.country, u.gender, u.traffic_source;
""")
dataset.show()

+--------------------+-----+
|            features|label|
+--------------------+-----+
|[36.0, 1.18756357...|    1|
|[47.0, -1.7219397...|    1|
|[27.0, 1.37825328...|    1|
|[56.0, -9.1453535...|    1|
|[47.0, 8.52158976...|    1|
|[24.0, 1.85268405...|    1|
|[28.0, 1.85268405...|    0|
|[51.0, 1.85268405...|    1|
|[20.0, 1.18756357...|    1|
|[27.0, -1.7219397...|    1|
|[19.0, -6.3182028...|    1|
|[60.0, 1.85268405...|    0|
|[59.0, -6.4130743...|    0|
|[24.0, 1.85268405...|    1|
|[45.0, -1.7219397...|    0|
|[41.0, 1.37825328...|    0|
|[44.0, 1.85268405...|    1|
|[52.0, 1.85268405...|    1|
|[29.0, -1.7219397...|    0|
|[50.0, -6.4130743...|    1|
+--------------------+-----+
only showing top 20 rows



# **Step 5: Perform ML Task**

This code trains a logistic regression model to predict user purchase behavior, with these steps:

**Feature Scaling:** StandardScaler scales the "features" column.

**Model Initialization:** LogisticRegression is set up to predict the binary "label" (purchase/no purchase), with hyperparameters for training.

**Pipeline Definition:** A Pipeline chains StandardScaler and LogisticRegression for streamlined scaling and training.

**Model Training:** `pipeline.fit(dataset)` trains the pipeline (scaling and then the model).

**Prediction:** `pipeline_model.transform(dataset)` generates predictions, and `transformed_dataset.show()` displays the results.

In short, this step scales features, trains a logistic regression model within a pipeline, and produces purchase predictions.

In [190]:
#Split Train and Test Data (80:20)
train_data, test_data = dataset.randomSplit([0.8, 0.2], seed=42)

In [191]:
# Initialize StandardScaler
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

# Initialize Logistic Regression model
lr = LogisticRegression(maxIter=20, learningRate=0.01, featuresCol="scaled_features", labelCol="label")

# Define pipeline
pipeline = Pipeline(stages=[scaler, lr])

# Fit the model
pipeline_model = pipeline.fit(train_data)

# Transform the dataset using the trained model
transformed_dataset = pipeline_model.transform(test_data)

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/usr/lib/spark/python/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 11) than that in driver 3.10, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.


In [ ]:
transformed_dataset.show()

# **Step 6: Evaluation**

This code evaluates the trained model's performance by:

**Initializing an Evaluator:** A BinaryClassificationEvaluator is set up to calculate the Area Under the Precision-Recall Curve (AUC-PR).

**Calculating AUC-PR:** The evaluate() method calculates the AUC-PR score using the model's predictions.

This step quantifies the model's ability to distinguish between the two classes (e.g., purchase/no purchase).


Further we will use NLP2SQL code generation to visualize the output

Prompt 1: Generate code to plot the Precision-Recall (PR) curve. Calculate precision and recall from the model's predictions and display the PR curve using a suitable plotting library.

Prompt 2: Generate code to create a confusion matrix visualization. Calculate the confusion matrix from the model's predictions and display it as a heatmap or a table with counts of true positives (TP), true negatives (TN), false positives (FP), and false negatives (FN).

In [ ]:
!pip install torcheval

In [ ]:
# Model evaluation
eva = BinaryClassificationEvaluator(metricName="areaUnderPR")
aucPR = eva.evaluate(transformed_dataset)
print(f"AUC PR: {aucPR}")

# **Step 7: Visualization**

Let's visualize the results to see how our model performs, and how it has predicted.

In [ ]:
# prompt: Generate code to plot the Precision-Recall (PR) curve. Calculate precision and recall from the model's predictions and display the PR curve using a suitable plotting library.

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
import pandas as pd

# Convert Spark DataFrame to Pandas DataFrame
transformed_pd = transformed_dataset.select("label", "prediction").toPandas()

# Calculate precision and recall
precision, recall, _ = precision_recall_curve(transformed_pd["label"], transformed_pd["prediction"])

# Plot the PR curve
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
# prompt: Generate code to create a confusion matrix visualization. Calculate the confusion matrix from the model's predictions and display it as a heatmap or a table with counts of true positives (TP), true negatives (TN), false positives (FP), and false negatives (FN).

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Assuming 'transformed_dataset' is a Spark DataFrame with 'label' and 'prediction' columns
predictions_pd = transformed_dataset.select('label', 'prediction').toPandas()

# Calculate the confusion matrix
cm = confusion_matrix(predictions_pd['label'], predictions_pd['prediction'])

# Create a heatmap visualization
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


# Display the confusion matrix as a table
cm_df = pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1'])
print("Confusion Matrix (Table):", cm_df)

# Extract TP, TN, FP, FN
tn, fp, fn, tp = cm.ravel()
print(f"True Positives (TP): {tp}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")

In [ ]:
predictions_pd.head()

In [ ]:
import seaborn as sns

# Convert feature list into individual columns
pdf = dataset.select("features", "label").toPandas()
feature_df = pd.DataFrame(pdf["features"].tolist(), columns=["age", "country_hash", "gender_hash", "traffic_source_hash"])

# Plot histograms
feature_df.hist(figsize=(10, 6), bins=30, color="dodgerblue", alpha=0.7)
plt.suptitle("Feature Distributions", fontsize=14)
plt.show()

In [199]:
import sys
print(sys.version)

3.10.12 (main, May 27 2025, 17:12:29) [GCC 11.4.0]


In [203]:
sys.path.insert(0, '/usr/bin/python3.11')

In [204]:
!python --version

Python 3.10.12


In [205]:
!echo $PATH

/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
